# Chapter 3: RNA-level enrichment reflects stronger functional selection compared to DNA
## Create AAV6-ML figures for Chapter 3 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 22/04/2026

Finished: ...

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'3_log2_enrichment'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

In [ ]:
# comare and corr two tables
def compare_two_tables(
    df1,
    df2,
    name1="sample_1",
    name2="sample_2",
    compare_cols=None,
    key_col="AA_sequence",
    log_scale_cols=None,
    add_diagonal=True,
    alpha=0.25,
    point_size=8,
    figsize_per_panel=(5.5, 5),
    save=False,
    output_path=None,
    file_name=None,
    show=True
):
    """
    Compare two tables across one or multiple numeric columns by merging on key_col.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Tables to compare
    name1, name2 : str
        Names shown in axis labels
    compare_cols : list of str
        Numeric columns to compare, e.g. ["Log2_enrichment", "RPM"]
    key_col : str
        Merge key, default = "AA_sequence"
    log_scale_cols : list of str
        Columns that should be plotted on log-log axes, e.g. ["RPM"]
    add_diagonal : bool
        Whether to add y=x diagonal
    alpha : float
        Point transparency
    point_size : int
        Scatter point size
    figsize_per_panel : tuple
        Width, height per subplot
    save : bool
        Whether to save figure
    output_path : str or Path
        Save directory
    file_name : str
        Output filename
    show : bool
        Whether to show plot

    Returns
    -------
    merged : pd.DataFrame
        Merged comparison table
    corr_df : pd.DataFrame
        Correlation summary table
    """

    if compare_cols is None:
        compare_cols = ["Log2_enrichment", "RPM"]

    if log_scale_cols is None:
        log_scale_cols = []

    # keep only needed columns
    cols1 = [key_col] + compare_cols
    cols2 = [key_col] + compare_cols

    a = df1[cols1].copy()
    b = df2[cols2].copy()

    # rename value columns
    a = a.rename(columns={col: f"{col}_{name1}" for col in compare_cols})
    b = b.rename(columns={col: f"{col}_{name2}" for col in compare_cols})

    # merge
    merged = a.merge(b, on=key_col, how="inner")

    # correlation results
    results = []

    # figure layout
    n = len(compare_cols)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows)
    )

    # make axes iterable
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).reshape(-1)

    for i, col in enumerate(compare_cols):
        ax = axes[i]

        xcol = f"{col}_{name1}"
        ycol = f"{col}_{name2}"

        sub = merged[[xcol, ycol]].dropna().copy()

        # correlations
        if len(sub) > 1:
            pearson_r, pearson_p = pearsonr(sub[xcol], sub[ycol])
            spearman_r, spearman_p = spearmanr(sub[xcol], sub[ycol])
        else:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_r, spearman_p = np.nan, np.nan

        results.append({
            "column": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "n_points": len(sub)
        })

        # scatter
        sns.scatterplot(
            data=sub,
            x=xcol,
            y=ycol,
            s=point_size,
            alpha=alpha,
            ax=ax
        )

        # log scaling if requested
        if col in log_scale_cols:
            sub_pos = sub[(sub[xcol] > 0) & (sub[ycol] > 0)].copy()
            ax.clear()
            sns.scatterplot(
                data=sub_pos,
                x=xcol,
                y=ycol,
                s=point_size,
                alpha=alpha,
                ax=ax
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            plot_data = sub_pos
        else:
            plot_data = sub

        # diagonal
        if add_diagonal and len(plot_data) > 0:
            lims = [
                min(plot_data[xcol].min(), plot_data[ycol].min()),
                max(plot_data[xcol].max(), plot_data[ycol].max())
            ]
            ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.6)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        ax.set_xlabel(f"{col} ({name1})")
        ax.set_ylabel(f"{col} ({name2})")
        ax.set_title(col)

        ax.text(
            0.05, 0.95,
            f"Pearson r = {pearson_r:.4f}\nSpearman ρ = {spearman_r:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        sns.despine(ax=ax)

    # remove unused axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        import os
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            col_string = "_".join(compare_cols)
            file_name = f"compare_{name1}_vs_{name2}_{col_string}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    corr_df = pd.DataFrame(results)

    return merged, corr_df

## 3. Script Functions

### 3.1. Violin plot of log2_enrichment in samples

In [ ]:
def violin_enrichment_per_type_test(
    table,
    order=None,
    n_samples=6,
    violin=False,
    box=False,
    title=True,
    save=False,
    file_name=None,
    output_path=None
):
    """
    Plot Log2 enrichment distributions across tissue/extraction groups,
    restricted to variants present in at least a given number of biological samples.

    The function creates violin plots and/or boxplots for the selected groups
    and annotates each category with the number of included variants (`n`).

    Parameters
    ----------
    table : pd.DataFrame
        Input table expected to contain at least the columns:
        ['AA_sequence', 'Extraction_type', 'Tissue', 'Log2_enrichment',
         'n_samples_present'].
    order : list of str, optional
        Order of plotted groups. If None, the default order is:
        ['gDNA_liver', 'cDNA_liver', 'gDNA_heart', 'cDNA_heart'].
    n_samples : int, default=6
        Minimum number of biological samples in which a variant must be present
        to be included in the plot.
    violin : bool, default=False
        Whether to draw violin plots.
    box : bool, default=False
        Whether to draw boxplots.
    title : bool, default=True
        Whether to show a plot title.
    save : bool, default=False
        Whether to save the figure.
    file_name : str, optional
        File name for saving the figure. If None, a default name is generated.
    output_path : str or Path, optional
        Output directory for saving the figure. Required if save=True.

    Returns
    -------
    None
        Displays the figure and optionally saves it to disk.

    Notes
    -----
    - Variants are filtered by `n_samples_present >= n_samples`.
    - Group labels are generated as `Extraction_type + "_" + Tissue`.
    - The number of included variants per group is shown above each category.
    """

    # ----------------------------------------------------------
    # Define plotting order if none is provided
    # ----------------------------------------------------------
    if order is None:
        order = ["gDNA_liver", "cDNA_liver", "gDNA_heart", "cDNA_heart"]

    # ----------------------------------------------------------
    # Prepare working copy and create combined group label
    # ----------------------------------------------------------
    df = table.copy()
    df["Type"] = df["Extraction_type"] + "_" + df["Tissue"]

    # ----------------------------------------------------------
    # Filter variants by minimum sample presence
    # ----------------------------------------------------------
    df = df.loc[
        (df["n_samples_present"] >= n_samples)
    ]

    # ----------------------------------------------------------
    # Keep only required columns and requested plot groups
    # ----------------------------------------------------------
    df = df[["AA_sequence", "Type", "Log2_enrichment"]].copy()
    df = df[df["Type"].isin(order)].copy()

    # ----------------------------------------------------------
    # Count number of included variants per plotted group
    # ----------------------------------------------------------
    n_per_group = (
        df.groupby("Type")["AA_sequence"]
        .nunique()
        .reindex(order, fill_value=0)
    )

    # ----------------------------------------------------------
    # Define color palette
    # ----------------------------------------------------------
    palette = {
        "gDNA_liver": "#4C5C68",
        "cDNA_liver": "#5DA5DA",
        "gDNA_heart": "#6B6D9C",
        "cDNA_heart": "#B276B2",
    }

    # ----------------------------------------------------------
    # Set plot style
    # ----------------------------------------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # ----------------------------------------------------------
    # Optional violin plot
    # ----------------------------------------------------------
    if violin:
        sns.violinplot(
            data=df,
            x="Type",
            y="Log2_enrichment",
            hue="Type",
            order=order,
            hue_order=order,
            palette=palette,
            dodge=False,
            legend=False,
            cut=0,
            inner="quartile",
            ax=ax
        )

    # ----------------------------------------------------------
    # Optional boxplot
    # ----------------------------------------------------------
    if box:
        sns.boxplot(
            data=df,
            x="Type",
            y="Log2_enrichment",
            order=order,
            width=0.3,
            showcaps=True,
            showfliers=False,
            boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.0, "zorder": 3},
            whiskerprops={"linewidth": 1.0, "zorder": 3},
            medianprops={"color": "black", "linewidth": 1.2, "zorder": 4},
            ax=ax
        )

    # ----------------------------------------------------------
    # Reference line for no enrichment
    # ----------------------------------------------------------
    ax.axhline(0, color="0.5", linestyle="--", linewidth=1)

    # ----------------------------------------------------------
    # Axis labels and optional title
    # ----------------------------------------------------------
    
    ax.set_ylabel("Log2 enrichment")
    ax.set_xlabel("")
    pretty_labels = {
        "gDNA_liver": "gDNA\nLiver",
        "cDNA_liver": "cDNA\nLiver",
        "gDNA_heart": "gDNA\nHeart",
        "cDNA_heart": "cDNA\nHeart",
    }
    
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([pretty_labels.get(x, x) for x in order])
    
    if title:
        ax.set_title(
            f"Log2 comparison of samples\nIncluded variants are present in at least {n_samples} samples",
            pad=18
        )

    # ----------------------------------------------------------
    # Add n labels above plot area
    # ----------------------------------------------------------
    for i, group in enumerate(order):
        ax.text(
            i,
            1.04,
            f"n = {n_per_group[group]:,}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=9
        )
    
    sns.despine(ax=ax)
    
    # gives more room for title + n labels above axes
    fig.tight_layout(rect=[0, 0, 1, 0.92])

    # ----------------------------------------------------------
    # Optional save
    # ----------------------------------------------------------
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"Log2_enrichment_comparison_variants_in_mind_{n_samples}_samples.png"
        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

### 3.2. KDE plot one per tissue, comparison gDNA vs cDNA

In [ ]:
# KDE plot for enrichment
def kde_gDNA_vs_cDNA(
    table, 
    tissue, 
    column,
    n_samples = 6,
    title = True,
    save = False,
    file_name = None,
    output_path = None
    
):
    
    df = table.copy()
    df = df[['AA_sequence', 'Tissue', 'Extraction_type', column, 'n_samples_present']]
    df = df.loc[
        (df['Tissue'] == tissue)
        ]
    

    df = df.loc[
    (df['n_samples_present'] >= n_samples)
    ]

    gDNA_values = df[df['Extraction_type'] == 'gDNA']
    cDNA_values = df[df['Extraction_type'] == 'cDNA']
    gDNA_variants = len(gDNA_values['AA_sequence'])
    cDNA_variants = len(cDNA_values['AA_sequence'])
    
    gDNA_color = "#4C5C68"
    cDNA_color = "#5DA5DA"

    plt.figure(figsize=(7, 4.6))

    sns.kdeplot(
        data=gDNA_values,
        x=column,
        fill=True,
        alpha=0.4,
        linewidth=1.5,
        color=gDNA_color,
        label=f"{gDNA_variants} gDNA variants"
    )

    sns.kdeplot(
        data=cDNA_values,
        x=column,
        fill=True,
        alpha=0.4,
        linewidth=1.5,
        color=cDNA_color,
        label=f"{cDNA_variants} cDNA variants"
    )
    if column == 'Log2_enrichment':
        plt.axvline(0, color="0.5", linestyle="--", linewidth=1)

    if column == 'Proportion':
        plt.xscale('log')

    plt.xlim(-10, 16)

    if title:
        plt.title(f'{tissue}\nPresent in mind. {n_samples} samples', fontsize = 12)
   
    plt.xlabel(column)
    plt.ylabel("Density")
    

    plt.legend(frameon=False)

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"KDE_log_enrichment_{tissue}_gDNA_vs_cDNA_in_mind_{n_samples}_samples.png"
        plt.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")
    
    plt.show()
    

### 3.3. Venn2 of overlap between top 10000 Log2_enrichment¶

In [ ]:
def venn2_top_enrichment_variants (table, tissue, n_top, n_samples = 0, title = True, save = False, output_path = None, file_name = None):

    # select samples I want to compare
    sample = (table [
        (table['Tissue'] == tissue) &
        (table['n_samples_present'] >= n_samples)
        ][["AA_sequence", 'Log2_enrichment', 'Extraction_type', 'n_samples_present']]).copy()

    cDNA_samples = sample[sample['Extraction_type'] == 'cDNA'].sort_values('Log2_enrichment', ascending = False).head(n_top).copy()
    cDNA_number = cDNA_samples.shape[0]
    if cDNA_number <= n_top:
        gDNA_samples = sample[sample['Extraction_type'] == 'gDNA'].sort_values('Log2_enrichment', ascending = False).head(cDNA_number).copy()
    else:
        gDNA_samples = sample[sample['Extraction_type'] == 'gDNA'].sort_values('Log2_enrichment', ascending = False).head(n_top).copy()
    
    # get AA_variants in set to compare them

    plt.figure(figsize=(7,4))
    v = venn2([set(gDNA_samples["AA_sequence"]), set(cDNA_samples["AA_sequence"])], set_labels=(f"gDNA {tissue}", f"cDNA {tissue}"))

    # set title
    if title:
        if cDNA_number <= n_top:
            plt.title(f"{tissue}\nOverlap of top {cDNA_number} Log2_enrichment AA variants for gDNA and cDNA\nIn min. {n_samples}/6 samples")
        else:
            plt.title(f"{tissue}\nOverlap of top {n_top} Log2_enrichment AA variants for gDNA and cDNA\nIn min. {n_samples}/6 samples")
    
    # Zahlen größer machen
    for text in v.subset_labels:
        if text:
            text.set_fontsize(18)
            text.set_fontweight("bold")
    
    # Set-Labels größer machen
    for text in v.set_labels:
        if text:
            text.set_fontsize(14)
    
    plt.tight_layout()

    # save file
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"Venn2_of_top_{cDNA_number}_log2_enrich_variants_in_{tissue}_gDNA_vs_cDNA_in_min_{n_samples}_samples.png"
        plt.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

### 3.4. Diagramm of Venn2 plot overlap between top 1 and all variants

In [ ]:
def Diagram_for_overlap_log2(
    table,
    tissues,
    n_samples=0,
    title=True,
    save=False,
    value = 'Log2_enrichment',
    output_path=None,
    file_name=None
):
    
    color_map = {
        "liver": "blue",
        "heart": "red"
    }

    fig, ax = plt.subplots(figsize=(6, 4))

    all_plot_values = []

    for tissue in tissues:
        # Samples auswählen
        sample = table[
            (table["Tissue"] == tissue) &
            (table["n_samples_present"] >= n_samples)
        ][["AA_sequence", value, "Extraction_type", "n_samples_present"]].copy()

        cDNA_samples = list(
            sample[sample["Extraction_type"] == "cDNA"]
            .sort_values(value, ascending=False)["AA_sequence"]
        )

        gDNA_samples = list(
            sample[sample["Extraction_type"] == "gDNA"]
            .sort_values(value, ascending=False)["AA_sequence"]
        )

        # maximale gemeinsame Länge
        n_max = min(len(cDNA_samples), len(gDNA_samples))
        
        log2_list = list(2 ** np.arange(1, int(np.log2(n_max)) + 1))
        log2_list = sorted(set(log2_list + [n_max]))
        plot_values = []

        for i in log2_list:
            cDNA_top = cDNA_samples[:i]
            gDNA_top = gDNA_samples[:i]

            overlap_fraction = len(set(cDNA_top) & set(gDNA_top)) / i
            plot_values.append(overlap_fraction * 100)

        all_plot_values.extend(plot_values)


        ax.plot(
            log2_list,
            plot_values,
            marker="o",
            linewidth=1.5,
            color=color_map.get(tissue, None),
            label=tissue
        )

    ax.set_xscale("log", base=10)

    ax.set_xlabel("Top N variants")
    ax.set_ylabel("Overlap [%]")

    if all_plot_values:
        ax.set_ylim(0, max(all_plot_values) + 1.1)

    if title:
        if len(tissues) == 1:
            ax.set_title(
                f"{tissues[0]}\nTop-N overlap of gDNA and cDNA variants\n"
                f"Variant present in min. {n_samples}/6 samples"
            )
        else:
            ax.set_title(
                f"Top-N overlap of gDNA and cDNA variants\n"
                f"Variant present in min. {n_samples}/6 samples"
            )

    ax.legend(frameon=False)
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            tissue_string = "_".join(tissues)
            file_name = f"overlap_log2_{tissue_string}_in_min_{n_samples}_samples.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    plt.show()

### 3.5. ECDF Log2 distribution between samples

In [ ]:
df_long.sort_values('Log2_enrichment', ascending = False)

In [ ]:
def distribution_ecdf(
    table,
    tissue,
    extraction,
    column,
    replicate,
    measure = 'median',
    no_pseudo = False,
    figsize = (7.5, 5.2),
    legend=True,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    
    if replicate == "biological":
        legend_name = "Data"
    elif replicate == "technical":
        legend_name = "Sample"
    else:
        raise ValueError("Replicate must be 'biological' or 'technical'")

    # -----------------------------
    # Prepare input library
    # -----------------------------
    input_library = dict_library[tissue].copy()
    input_library[legend_name] = "input_library"

    # -----------------------------
    # Prepare selected sample data
    # -----------------------------
    df_sample = table[
        (table["Replicate"] == replicate) &
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][["AA_sequence", "Sample", "Mouse_ID", f'{column}', "Pseudo"]].copy()
    
    # usage of pseudo variants or not
    if no_pseudo:
        df_sample = df_sample[df_sample['Pseudo'] == 0]
        
    if replicate == "biological":
        df_sample = df_sample.rename(columns={"Mouse_ID": "Data"})

    if column == 'Proportion':
        df_sample = df_sample[df_sample['Pseudo'] == 0]

    # -----------------------------
    # Concat input + selected samples
    # -----------------------------
    df = pd.concat([input_library, df_sample], axis=0, ignore_index=True)

    # keep valid values only
    df = df[df[column].notna()].copy()

    # define x-range from raw data
    x_min = table[column].min()
    x_max = table[column].max()

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=figsize)

    # -----------------------------
    # Define colors
    # -----------------------------
    color_palette = {
        "input_library": "#BDBDBD",  # grey
        "f1": "#9ECAE1",             # blue shades
        "f2": "#6BAED6",
        "f3": "#3182BD",
        "m1": "#D4B9DA",             # violet shades
        "m2": "#C994C7",
        "m3": "#88419D"
    }

    # -----------------------------
    # Define plotting order
    # -----------------------------
    present_samples = [x for x in df[legend_name].dropna().unique() if x != "input_library"]

    ordered_samples = [x for x in MOUSE_ID if x in present_samples] + \
                      [x for x in present_samples if x not in MOUSE_ID]

    sample_order = ["input_library"] + ordered_samples

    # -----------------------------
    # Plot ECDFs
    # -----------------------------
    for data_name in sample_order:
        sub = df[df[legend_name] == data_name].copy()
        if sub.empty:
            continue

        color = color_palette.get(data_name, "#4C72B0")

        # input in background, full opacity
        if data_name == "input_library":
            alpha_val = 1.0
            lw = 2.2
            z = 1
        else:
            alpha_val = 0.5
            lw = 2.0
            z = 2

        sns.ecdfplot(
            data=sub,
            x=column,
            ax=ax,
            color=color,
            alpha=alpha_val,
            linewidth=lw,
            label=data_name,
            zorder=z
        )

    # -----------------------------
    # Reference lines
    # -----------------------------

    if measure == 'mean':
        val= df_sample[column].mean()
    if measure == 'median':
        val= df_sample[column].median()

    # for proportion: only mean line
    # for log2_enrichment: zero line + mean line
    line_handles = []

    if column.startswith("Log2"):
        line_zero = ax.axvline(
            0,
            linestyle="--",
            linewidth=1.5,
            color="red",
            alpha=0.9,
            label="No enrichment = 0"
        )
        line_handles.append(line_zero)

    line_mean = ax.axvline(
        val,
        linestyle="--",
        linewidth=1.5,
        color="black",
        alpha=0.9,
        label=f"Mean = {val:.2e}" if measure == "mean" else f"Median = {val:.2f}"
    )

    line_handles.append(line_mean)

    # -----------------------------
    # Axes
    # -----------------------------
    ax.set_xlim(x_min, x_max)

    if column == "Proportion":
        ax.set_xscale("log")
        ax.set_xlabel("Variant proportion")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution of AA-sequence proportions"
    else:
        ax.set_xlabel(f"Variant {column}")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution of {column}"

    ax.set_ylabel(ylab)

    if title:
        ax.set_title(title_text)

    # -----------------------------
    # Legend
    # -----------------------------
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))  # remove duplicates, preserve last occurrence
        ax.legend(
            unique.values(),
            unique.keys(),
            loc="lower right",
            frameon=True,
            ncol=3
        )
    else:
        ref_handles = []
        ref_labels = []

        ref_handles.append(line_mean)
        ref_labels.append(f"Mean = {val:.2e}" if measure == "mean" else f"Median = {val:.2f}")

        ax.legend(
            ref_handles,
            ref_labels,
            loc="lower right",
            frameon=True
        )
        

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"ECDF_{column}_{extraction}_{tissue}_{replicate}.png"
        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. Violin plot mean between biological Replicate

In [ ]:

violin_enrichment_per_type_test (
    df_pooled, 
    violin = True, 
    box = True, 
    n_samples=2, 
    title=False,
    save = True, 
    output_path=figures_dir/'1_violin',
    file_name=f'violin_box_log2_enrichment_variant_in_min_2_samples.png'
)

    

### 5.2. KDE plot for liver and heart (DNA and RNA in one plot, x = enrichment, y = n variant)

In [ ]:
for tissue in TISSUE:
    kde_gDNA_vs_cDNA (
        df_pooled, 
        tissue, 
        column = 'Log2_enrichment', 
        n_samples = 2, 
        title=True,
        save = True, 
        output_path=figures_dir/'2_KDE',
        file_name=f'KDE_log_enrichmentin_{tissue}_mind_2_samples'
        
    )

### 5.3.Venn2 of overlap between top 10000 Log2_enrichment

In [ ]:
for tissue in TISSUE:
    venn2_top_enrichment_variants (
        df_pooled,
        tissue,
        10000,
        n_samples=2,
        title=False,
        save = False,
        output_path=figures_dir/'3_Venn2_log2',
        file_name=f'Venn2_{tissue}_min_2_samples_big_text_no_title'
    )

### 5.4. Diagramm of Venn2 plot overlap between top 1 and all variants

In [ ]:
Diagram_for_overlap_log2 (
    table=df_pooled, 
    tissues=['liver', 'heart'], 
    n_samples=2, 
    value= 'Log2_enrichment_old',
    title=True,
    save = True, 
    output_path=figures_dir/'4_Diagram_of_overlap',
    file_name='overlap_log2_old_min_2_samples'
)

### 5.5. ECDF Log2_enrichment_gDNA_to_cDNA distribution between samples

In [ ]:
%%time
distribution_ecdf (
    table = df_long, 
    tissue = 'heart',
    extraction = 'cDNA', 
    column = 'Log2_enrich_gDNA_to_cDNA', 
    replicate = 'biological', 
    measure='median',
    figsize=(8, 5),
    title=False,
    legend=True,
    no_pseudo=True,
    save = True, 
    output_path = figures_dir/'5_ECDF'/'Log2_enrichment_gDNA_to_cDNA'
)

In [ ]:
%%time
distribution_ecdf (
    table = df_long, 
    tissue = 'heart',
    extraction = 'cDNA', 
    column = 'Log2_enrichment', 
    replicate = 'biological', 
    measure='median',
    title=False,
    legend=True,
    no_pseudo=True,
    save = True, 
    output_path = figures_dir/'5_ECDF'/'no_pseudo'
)